# Tennis Match Outcome Predictor — Exploration Notebook

This notebook walks through:
1. Loading and exploring the raw ATP data
2. Visualising distributions of key features
3. Checking the engineered feature matrix
4. Quick model comparison
5. Roland Garros 2025 results analysis

In [ ]:
import sys
from pathlib import Path

# Make sure project root is on the path
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

## 1. Raw Data Overview

In [ ]:
raw_dir = ROOT / 'data' / 'raw'
csv_files = sorted(raw_dir.glob('atp_matches_[12][0-9][0-9][0-9].csv'))
print(f'Found {len(csv_files)} year CSV files')

# Load a sample year
if csv_files:
    sample = pd.read_csv(csv_files[-1])  # most recent year
    print(f'\nMost recent file: {csv_files[-1].name}  ({len(sample)} rows)')
    sample.head()

In [ ]:
# Load all years quickly using the cleaner module
from preprocessing.cleaner import load_all_matches, clean

raw_df = load_all_matches(raw_dir)
print(f'Total raw rows: {len(raw_df):,}')
raw_df.dtypes

In [ ]:
clean_df = clean(raw_df)
print(f'After cleaning: {len(clean_df):,} rows')
clean_df['date'] = pd.to_datetime(clean_df['date'])
clean_df.describe()

## 2. Data Distributions

In [ ]:
# Matches per year
clean_df['year'] = clean_df['date'].dt.year
matches_per_year = clean_df.groupby('year').size()
matches_per_year.plot(kind='bar', title='ATP Main Tour Matches per Year', figsize=(16, 4))
plt.tight_layout()
plt.show()

In [ ]:
# Surface distribution
if 'surface' in clean_df.columns:
    clean_df['surface'].value_counts().plot(kind='bar', title='Matches by Surface')
    plt.tight_layout()
    plt.show()

In [ ]:
# Ranking distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
if 'winner_rank' in clean_df.columns:
    clean_df['winner_rank'].clip(upper=200).hist(bins=50, ax=axes[0])
    axes[0].set_title('Winner Rank Distribution (clipped at 200)')
if 'loser_rank' in clean_df.columns:
    clean_df['loser_rank'].clip(upper=200).hist(bins=50, ax=axes[1])
    axes[1].set_title('Loser Rank Distribution (clipped at 200)')
plt.tight_layout()
plt.show()

## 3. Engineered Features

In [ ]:
features_csv = ROOT / 'data' / 'processed' / 'features.csv'
if features_csv.exists():
    feat_df = pd.read_csv(features_csv)
    feat_df['date'] = pd.to_datetime(feat_df['date'])
    print(f'Feature matrix: {feat_df.shape}')
    feat_df.head()
else:
    print('features.csv not found — run preprocessing/feature_engineering.py first.')

In [ ]:
if features_csv.exists():
    # ELO distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    feat_df['elo_p1'].hist(bins=50, ax=axes[0])
    axes[0].set_title('ELO Distribution — P1')
    feat_df['elo_diff'].hist(bins=50, ax=axes[1], color='orange')
    axes[1].set_title('ELO Difference (P1 − P2)')
    plt.tight_layout()
    plt.show()

In [ ]:
if features_csv.exists():
    # Feature correlation heatmap
    numeric_cols = feat_df.select_dtypes(include='number').columns.tolist()
    corr = feat_df[numeric_cols].corr()
    plt.figure(figsize=(14, 10))
    sns.heatmap(corr, cmap='coolwarm', center=0, fmt='.2f', annot=False)
    plt.title('Feature Correlation Matrix')
    plt.tight_layout()
    plt.show()

## 4. Model Performance

In [ ]:
eval_csv = ROOT / 'models' / 'evaluation_results.csv'
if eval_csv.exists():
    eval_df = pd.read_csv(eval_csv)
    print('Model Evaluation Results:')
    display(eval_df.set_index('model'))
else:
    print('evaluation_results.csv not found — run models/train.py first.')

## 5. Roland Garros 2025 Results

In [ ]:
rg_real = pd.read_csv(ROOT / 'results' / 'roland_garros_2025.csv')
print(f'Real results: {len(rg_real)} matches')
rg_real

In [ ]:
predictions_csv = ROOT / 'results' / 'roland_garros_2025_predictions.csv'
if predictions_csv.exists():
    preds = pd.read_csv(predictions_csv)
    print(f'Predictions: {len(preds)} matches')
    display(preds)
else:
    print('Predictions not found — run models/predict.py first.')

In [ ]:
comparison_csv = ROOT / 'results' / 'roland_garros_2025_comparison.csv'
if comparison_csv.exists():
    comp = pd.read_csv(comparison_csv)
    round_acc = comp.groupby('round')['correct'].mean().reset_index()
    round_acc.columns = ['Round', 'Accuracy']
    round_acc['Accuracy'] = (round_acc['Accuracy'] * 100).round(1)
    print('Accuracy per round:')
    display(round_acc)

    # Bar chart
    round_acc.plot(x='Round', y='Accuracy', kind='bar',
                   title='RG 2025 Prediction Accuracy per Round (%)',
                   ylim=(0, 100), legend=False)
    plt.axhline(50, linestyle='--', color='red', label='Random baseline')
    plt.legend()
    plt.tight_layout()
    plt.show()